# AVONET Dataset — Cross-Walk & Feature Engineering

**Dataset:** AVONET (Tobias et al., 2022)  
**Sources used:**
- `AVONET_Raw_Data.csv` — individual-level measurements (90,371 rows)
- `AVONET1_BirdLife.csv` — species-level summary (BirdLife taxonomy)
- `AVONET2_eBird.xlsx` — species-level summary (eBird taxonomy)
- `AVONET3_BirdTree.xlsx` — species-level summary (BirdTree taxonomy)
- `BirdLife_BirdTree_Taxonomy_Relation.csv` — crosswalk between BirdLife and BirdTree names

---

## Goals

1. **Species ID Mapping** — Build a clean, deduplicated table linking each `Avibase.ID` to its species name across all three taxonomies (BirdLife, eBird, BirdTree), along with family and order from each.
2. **Morphological Feature Averages** — For each species, compute overall, male, and female averages for all 10 morphological traits from the raw individual-level data.
3. **Enrich with BirdLife ecology data** — Add species-level metadata (mass, habitat, migration, trophic level, range, etc.) from the BirdLife summary table.

## 0 — Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/raw/Avonet")

# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)
pd.set_option("display.max_colwidth", 100)

# Suppress mixed-type warnings from messy raw data
import warnings
warnings.filterwarnings("ignore")

---
## 1 — Load Raw Data

The raw dataset contains one row per individual specimen measurement.  
The same species (`Avibase.ID`) can appear in thousands of rows.  
We keep only the first 26 columns — the rest are metadata we don't need.

In [27]:
df_raw = pd.read_csv(BASE / "core/AVONET_Raw_Data.csv", encoding="latin1")
df_raw = df_raw.iloc[:, :26]  # keep only measurement columns

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (90371, 26)


,Avibase.ID,Species1_BirdLife,Species2_eBird,eBird.species.group,Species3_BirdTree,Data.type,Source,Specimen.number,Sex,Age,Locality,Country_WRI,Country,Beak.Length_Culmen,Beak.Length_Nares,Beak.Width,Beak.Depth,Tarsus.Length,Wing.Length,Kipps.Distance,Secondary1,Hand-wing.Index,Tail.Length,Measurer,Protocol,Publication
0,AVIBASE-B3F5E5E2,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,1.0,NHMUK,11.4.1890,F,0.0,NaN,Guatemala,NaN,14.8,10.5,1.2,2.0,7.1,45.0,30.0,15.0,66.7,27.0,NJTA,1.0,NaN
1,AVIBASE-B3F5E5E2,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,1.0,NHMUK,12.23.1890,F,0.0,NaN,Guatemala,NaN,13.8,9.5,1.0,1.5,8.0,45.0,31.0,14.0,68.9,30.0,NJTA,1.0,NaN
2,AVIBASE-B3F5E5E2,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,1.0,NHMUK,1913.3.20.972,M,0.0,NaN,Guatemala,NaN,11.8,9.4,1.0,1.8,7.2,47.0,31.0,16.0,66.0,29.0,NJTA,1.0,NaN
3,AVIBASE-B3F5E5E2,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,1.0,NHMUK,87.3.22.1574,M,0.0,NaN,Guatemala,NaN,11.8,8.6,1.0,1.7,4.5,47.0,30.0,17.0,63.8,30.0,NJTA,1.0,NaN
4,AVIBASE-B3F5E5E2,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,Abeillia abeillei,1.0,NHMUK,NaN,M,0.0,NaN,NaN,NaN,12.8,NaN,1.7,1.4,4.9,51.0,NaN,NaN,NaN,31.0,GT,0.0,NaN


In [28]:
# Work on a copy so the raw data is always preserved
df = df_raw.copy()

---
## 2 — Species ID Mapping

### Problem
The raw data has many rows per species, and some rows have null values in the species name columns (`Species1_BirdLife`, `Species2_eBird`, `Species3_BirdTree`) while other rows for the same `Avibase.ID` have the values filled in.

A naive `drop_duplicates()` would either keep rows with nulls or drop rows that actually have useful name data.

### Solution: Null-coalescing groupby
Group by `Avibase.ID` and for each column take the **first non-null value** found across all rows in that group. This maximises data coverage — if any row has the species name, we keep it.

In [29]:
# For each Avibase.ID, pick the first non-null value per column
mapping = (
    df.groupby("Avibase.ID", as_index=False)
    .agg(lambda x: x.dropna().iloc[0] if x.notna().any() else None)
)

print(f"Original rows : {len(df):,}")
print(f"Unique species: {len(mapping):,}")

Original rows : 90,371
Unique species: 11,237


In [30]:
# Extract the 4 ID columns into a clean identification table
df_Identification = mapping[["Avibase.ID", "Species1_BirdLife", "Species2_eBird", "Species3_BirdTree"]]
df_Identification.head()

,Avibase.ID,Species1_BirdLife,Species2_eBird,Species3_BirdTree
0,AVIBASE-000E026B,Otus everetti,Otus everetti,Otus megalotis
1,AVIBASE-00124D98,Zenaida macroura,Zenaida macroura,Zenaida macroura
2,AVIBASE-00146574,None,Treron calvus,Treron calvus
3,AVIBASE-0018A99B,Bubo cinerascens,Bubo cinerascens,Bubo cinerascens
4,AVIBASE-002105C7,Treron sieboldii,Treron sieboldii,Treron sieboldii


In [31]:
# QC: how many nulls remain in the ID table?
# Some species genuinely don't have a name in every taxonomy
df_Identification.isnull().sum()

Avibase.ID            0
Species1_BirdLife    63
Species2_eBird       34
Species3_BirdTree    30
dtype: int64

---
## 3 — Morphological Feature Averages (Overall, Male, Female)

### Goal
For each `Avibase.ID`, compute three averages per morphological feature:
- `_avg` — all individuals combined
- `_avg_m` — males only
- `_avg_f` — females only

### Rules applied
| Scenario | Handling |
|---|---|
| `Avibase.ID` is null | Row ignored entirely |
| Feature value is null | Row skipped for that feature; ID still kept in output |
| `Sex` is null (unsexed) | Counted in overall avg; excluded from M/F |
| No M or F data for an ID | `_avg_m` / `_avg_f` stay `NaN` |
| ID exists but zero measurements | Still appears in output with all `NaN` |

In [32]:
def compute_sex_averages(df, feature_cols):
    """
    For each Avibase.ID, compute overall, male, and female averages
    for each feature in feature_cols.

    All valid Avibase IDs are preserved in the output even if no
    measurement data exists for a given feature.

    Parameters
    ----------
    df           : raw dataframe (individual-level measurements)
    feature_cols : list of numeric column names to average

    Returns
    -------
    DataFrame with columns:
        Avibase.ID | feat_avg | feat_avg_M | feat_avg_F | ...
    """
    # Drop rows with no Avibase.ID — nothing to group on
    df_valid = df.dropna(subset=["Avibase.ID"])

    # Anchor the output on all unique IDs so no species goes missing
    result = df_valid[["Avibase.ID"]].drop_duplicates().reset_index(drop=True)

    for col in feature_cols:
        # Ignore null measurements for this feature (but keep the ID)
        df_feat = df_valid.dropna(subset=[col])

        overall = (
            df_feat
            .groupby("Avibase.ID")[col]
            .mean()
            .rename(f"{col}_avg")
        )
        male = (
            df_feat[df_feat["Sex"] == "M"]
            .groupby("Avibase.ID")[col]
            .mean()
            .rename(f"{col}_avg_M")
        )
        female = (
            df_feat[df_feat["Sex"] == "F"]
            .groupby("Avibase.ID")[col]
            .mean()
            .rename(f"{col}_avg_F")
        )

        # Left join keeps all IDs — missing data becomes NaN naturally
        result = (
            result
            .join(overall, on="Avibase.ID")
            .join(male,    on="Avibase.ID")
            .join(female,  on="Avibase.ID")
        )

    return result

In [33]:
# All 10 morphological features to average
features = [
    "Beak.Length_Culmen",
    "Beak.Length_Nares",
    "Beak.Width",
    "Beak.Depth",
    "Tarsus.Length",
    "Wing.Length",
    "Kipps.Distance",
    "Secondary1",
    "Hand-wing.Index",
    "Tail.Length",
]

output = compute_sex_averages(df, features)

print(f"Avibase IDs : {len(output):,}")
print(f"Columns     : {len(output.columns)}")
output.head()

Avibase IDs : 11,237
Columns     : 31


,Avibase.ID,Beak.Length_Culmen_avg,Beak.Length_Culmen_avg_M,Beak.Length_Culmen_avg_F,Beak.Length_Nares_avg,Beak.Length_Nares_avg_M,Beak.Length_Nares_avg_F,Beak.Width_avg,Beak.Width_avg_M,Beak.Width_avg_F,Beak.Depth_avg,Beak.Depth_avg_M,Beak.Depth_avg_F,Tarsus.Length_avg,Tarsus.Length_avg_M,Tarsus.Length_avg_F,Wing.Length_avg,Wing.Length_avg_M,Wing.Length_avg_F,Kipps.Distance_avg,Kipps.Distance_avg_M,Kipps.Distance_avg_F,Secondary1_avg,Secondary1_avg_M,Secondary1_avg_F,Hand-wing.Index_avg,Hand-wing.Index_avg_M,Hand-wing.Index_avg_F,Tail.Length_avg,Tail.Length_avg_M,Tail.Length_avg_F
0,AVIBASE-B3F5E5E2,13.000000,12.133333,14.30,9.500,9.00,10.000000,1.1800,1.233333,1.100000,1.680,1.633333,1.750000,6.340000,5.533333,7.550000,47.000000,48.333333,45.000000,30.500,30.5,30.500000,15.500,16.50,14.50,66.350,64.90,67.800000,29.400,30.000000,28.500000
1,AVIBASE-684016CB,8.980000,9.066667,8.85,4.900,4.90,4.900000,2.7400,2.666667,2.850000,2.340,2.366667,2.300000,15.280000,15.433333,15.050000,43.400000,44.166667,42.250000,5.700,5.8,5.600000,37.300,37.95,36.65,13.275,13.25,13.300000,38.700,38.833333,38.500000
2,AVIBASE-0FD03AFF,8.950000,8.800000,9.00,4.675,4.50,4.733333,2.6000,2.700000,2.566667,2.325,2.500000,2.266667,16.525000,16.800000,16.433333,47.125000,48.000000,46.833333,7.600,8.4,7.333333,39.525,39.60,39.50,16.125,17.50,15.666667,47.500,48.000000,47.333333
3,AVIBASE-A8F5D21F,12.320000,12.266667,12.40,7.375,7.35,7.400000,3.1200,3.100000,3.150000,2.840,2.966667,2.650000,16.580000,16.766667,16.300000,51.200000,52.333333,49.500000,6.475,6.7,6.250000,44.525,45.80,43.25,12.700,12.75,12.650000,42.400,44.333333,39.500000
4,AVIBASE-50A42775,37.116667,38.883333,35.35,16.625,17.20,16.050000,9.2625,9.275000,9.250000,10.250,11.300000,9.200000,65.816667,68.500000,63.133333,337.181818,343.400000,332.000000,52.500,52.5,52.500000,283.100,291.50,277.50,15.660,15.30,15.900000,290.375,300.333333,280.416667


---
## 4 — Enrich with BirdLife Species-Level Data

The `AVONET1_BirdLife.csv` table contains one row per species with ecology and geography data (mass, habitat, migration, trophic level, range size, etc.).

We join this onto our `output` table using `Avibase.ID` as the key.

**Note:** The BirdLife table uses `Avibase.ID1` — we rename it first to match.

In [34]:
df_birdLife = pd.read_csv(BASE / "core/AVONET1_BirdLife.csv", encoding="latin1")

# Standardise the join key name
df_birdLife.rename(columns={"Avibase.ID1": "Avibase.ID"}, inplace=True)

print(f"BirdLife shape: {df_birdLife.shape}")
df_birdLife.head()

BirdLife shape: (11009, 37)


,Sequence,Species1,Family1,Order1,Avibase.ID,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,Beak.Width,Beak.Depth,Tarsus.Length,Wing.Length,Kipps.Distance,Secondary1,Hand-Wing.Index,Tail.Length,Mass,Mass.Source,Mass.Refs.Other,Inference,Traits.inferred,Reference.species,Habitat,Habitat.Density,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Min.Latitude,Max.Latitude,Centroid.Latitude,Centroid.Longitude,Range.Size
0,3103.0,Accipiter albogularis,Accipitridae,Accipitriformes,AVIBASE-BBB59880,5,2,0,3,4,27.7,17.8,10.6,14.7,62.0,235.2,81.8,159.5,33.9,169.0,248.8,Dunning,NaN,NO,NaN,NaN,Forest,1,2.0,Carnivore,Vertivore,Insessorial,-11.73,-4.02,-8.15,158.49,37461.21
1,3090.0,Accipiter badius,Accipitridae,Accipitriformes,AVIBASE-1A0ECB6E,10,4,6,0,8,20.6,12.1,8.8,11.6,43.0,186.7,62.5,127.4,32.9,140.6,131.2,Dunning,NaN,NO,NaN,NaN,Shrubland,2,3.0,Carnivore,Vertivore,Insessorial,-29.47,46.39,8.23,44.98,22374973.00
2,3125.0,Accipiter bicolor,Accipitridae,Accipitriformes,AVIBASE-ADBE44E1,11,4,5,2,8,25.0,13.7,8.6,12.7,58.1,229.6,56.6,174.8,24.6,186.3,287.5,Dunning,NaN,NO,NaN,NaN,Woodland,2,2.0,Carnivore,Vertivore,Generalist,-55.72,23.73,-10.10,-59.96,14309701.27
3,3116.0,Accipiter brachyurus,Accipitridae,Accipitriformes,AVIBASE-68BF920B,4,4,0,0,3,22.5,14.0,8.9,11.9,61.2,202.2,64.1,138.1,31.7,140.8,142.0,Dunning,NaN,NO,NaN,NaN,Forest,1,2.0,Carnivore,Vertivore,Insessorial,-6.31,-4.08,-5.45,150.68,35580.71
4,3092.0,Accipiter brevipes,Accipitridae,Accipitriformes,AVIBASE-8492E4B7,8,4,4,0,4,21.1,12.1,8.7,11.1,46.4,217.6,87.8,129.9,40.2,153.5,186.5,Dunning,NaN,NO,NaN,NaN,Forest,1,3.0,Carnivore,Vertivore,Generalist,31.19,55.86,45.24,45.33,2936751.80


In [35]:
def merge_by_avibase(our_dataset, other_dataset, feature):
    """
    Left-join a single feature from another dataset into ours using Avibase.ID.

    - IDs not found in other_dataset  → NaN (kept)
    - IDs found but value is null     → NaN (kept)
    - Row order of our_dataset preserved

    Parameters
    ----------
    our_dataset  : main dataframe
    other_dataset: source dataframe (must have Avibase.ID column)
    feature      : column name to bring in

    Returns
    -------
    our_dataset with the new column appended
    """
    return our_dataset.merge(
        other_dataset[["Avibase.ID", feature]],
        on="Avibase.ID",
        how="left"
    )

In [36]:
# Ecology and geography columns to bring in from BirdLife
birdlife_cols = [
   "Total.individuals",
    "Female",
    "Male",
    "Mass",
    "Mass.Source",
    "Inference",
    "Habitat",
    "Habitat.Density",
    "Migration",
    "Trophic.Level",
    "Trophic.Niche",
    "Primary.Lifestyle",
    "Min.Latitude",
    "Max.Latitude",
    "Centroid.Latitude",
    "Centroid.Longitude",
    "Range.Size",
]

for col in birdlife_cols:
    output = merge_by_avibase(output, df_birdLife, col)

print(f"Rows    : {len(output):,}")
print(f"Columns : {len(output.columns)}")
output.head()

Rows    : 11,237
Columns : 48


,Avibase.ID,Beak.Length_Culmen_avg,Beak.Length_Culmen_avg_M,Beak.Length_Culmen_avg_F,Beak.Length_Nares_avg,Beak.Length_Nares_avg_M,Beak.Length_Nares_avg_F,Beak.Width_avg,Beak.Width_avg_M,Beak.Width_avg_F,Beak.Depth_avg,Beak.Depth_avg_M,Beak.Depth_avg_F,Tarsus.Length_avg,Tarsus.Length_avg_M,Tarsus.Length_avg_F,Wing.Length_avg,Wing.Length_avg_M,Wing.Length_avg_F,Kipps.Distance_avg,Kipps.Distance_avg_M,Kipps.Distance_avg_F,Secondary1_avg,Secondary1_avg_M,Secondary1_avg_F,Hand-wing.Index_avg,Hand-wing.Index_avg_M,Hand-wing.Index_avg_F,Tail.Length_avg,Tail.Length_avg_M,Tail.Length_avg_F,Total.individuals,Female,Male,Mass,Mass.Source,Inference,Habitat,Habitat.Density,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Min.Latitude,Max.Latitude,Centroid.Latitude,Centroid.Longitude,Range.Size
0,AVIBASE-B3F5E5E2,13.000000,12.133333,14.30,9.500,9.00,10.000000,1.1800,1.233333,1.100000,1.680,1.633333,1.750000,6.340000,5.533333,7.550000,47.000000,48.333333,45.000000,30.500,30.5,30.500000,15.500,16.50,14.50,66.350,64.90,67.800000,29.400,30.000000,28.500000,5.0,2.0,3.0,2.7,Dunning,NO,Forest,2.0,1.0,Herbivore,Nectarivore,Aerial,12.78,19.77,15.29,-89.79,144610.40
1,AVIBASE-684016CB,8.980000,9.066667,8.85,4.900,4.90,4.900000,2.7400,2.666667,2.850000,2.340,2.366667,2.300000,15.280000,15.433333,15.050000,43.400000,44.166667,42.250000,5.700,5.8,5.600000,37.300,37.95,36.65,13.275,13.25,13.300000,38.700,38.833333,38.500000,5.0,2.0,3.0,4.8,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,14.97,31.56,24.78,105.57,1704650.60
2,AVIBASE-0FD03AFF,8.950000,8.800000,9.00,4.675,4.50,4.733333,2.6000,2.700000,2.566667,2.325,2.500000,2.266667,16.525000,16.800000,16.433333,47.125000,48.000000,46.833333,7.600,8.4,7.333333,39.525,39.60,39.50,16.125,17.50,15.666667,47.500,48.000000,47.333333,4.0,3.0,1.0,4.7,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,20.19,31.12,26.70,95.58,909217.05
3,AVIBASE-A8F5D21F,12.320000,12.266667,12.40,7.375,7.35,7.400000,3.1200,3.100000,3.150000,2.840,2.966667,2.650000,16.580000,16.766667,16.300000,51.200000,52.333333,49.500000,6.475,6.7,6.250000,44.525,45.80,43.25,12.700,12.75,12.650000,42.400,44.333333,39.500000,5.0,2.0,3.0,6.5,Dunning,NO,Shrubland,1.0,1.0,Carnivore,Invertivore,Insessorial,-8.11,28.81,9.89,105.20,2493437.47
4,AVIBASE-50A42775,37.116667,38.883333,35.35,16.625,17.20,16.050000,9.2625,9.275000,9.250000,10.250,11.300000,9.200000,65.816667,68.500000,63.133333,337.181818,343.400000,332.000000,52.500,52.5,52.500000,283.100,291.50,277.50,15.660,15.30,15.900000,290.375,300.333333,280.416667,12.0,6.0,6.0,1405.1,Dunning,NO,Forest,1.0,1.0,Herbivore,Frugivore,Insessorial,-13.43,11.15,-2.79,-75.23,139314.36


---
## 5 — Rename Columns (Clean Names)

Convert all column names to clean `snake_case` that is short but clear to a first-time viewer.

In [37]:
output.rename(columns={
    # ── Identity
    "Avibase.ID"                : "avibase_id",

    # ── Beak Culmen Length
    "Beak.Length_Culmen_avg"    : "beak_culmen_avg",
    "Beak.Length_Culmen_avg_M"  : "beak_culmen_avg_m",
    "Beak.Length_Culmen_avg_F"  : "beak_culmen_avg_f",

    # ── Beak Nares Length
    "Beak.Length_Nares_avg"     : "beak_nares_avg",
    "Beak.Length_Nares_avg_M"   : "beak_nares_avg_m",
    "Beak.Length_Nares_avg_F"   : "beak_nares_avg_f",

    # ── Beak Width
    "Beak.Width_avg"            : "beak_width_avg",
    "Beak.Width_avg_M"          : "beak_width_avg_m",
    "Beak.Width_avg_F"          : "beak_width_avg_f",

    # ── Beak Depth
    "Beak.Depth_avg"            : "beak_depth_avg",
    "Beak.Depth_avg_M"          : "beak_depth_avg_m",
    "Beak.Depth_avg_F"          : "beak_depth_avg_f",

    # ── Tarsus Length
    "Tarsus.Length_avg"         : "tarsus_avg",
    "Tarsus.Length_avg_M"       : "tarsus_avg_m",
    "Tarsus.Length_avg_F"       : "tarsus_avg_f",

    # ── Wing Length
    "Wing.Length_avg"           : "wing_len_avg",
    "Wing.Length_avg_M"         : "wing_len_avg_m",
    "Wing.Length_avg_F"         : "wing_len_avg_f",

    # ── Kipps Distance
    "Kipps.Distance_avg"        : "kipps_avg",
    "Kipps.Distance_avg_M"      : "kipps_avg_m",
    "Kipps.Distance_avg_F"      : "kipps_avg_f",

    # ── Secondary Feather
    "Secondary1_avg"            : "secondary_avg",
    "Secondary1_avg_M"          : "secondary_avg_m",
    "Secondary1_avg_F"          : "secondary_avg_f",

    # ── Hand-wing Index
    "Hand-wing.Index_avg"       : "hwi_avg",
    "Hand-wing.Index_avg_M"     : "hwi_avg_m",
    "Hand-wing.Index_avg_F"     : "hwi_avg_f",

    # ── Tail Length
    "Tail.Length_avg"           : "tail_avg",
    "Tail.Length_avg_M"         : "tail_avg_m",
    "Tail.Length_avg_F"         : "tail_avg_f",

    # ── Mass
    "Mass"                      : "mass_avg",
    "Mass.Source"               : "mass_source",

    # ── Sample Counts
    "Total.individuals"         : "total_individuals",
    "Female"                    : "female_count",
    "Male"                      : "male_count",

    # ── Ecology
    "Inference"                 : "inference",
    "Habitat"                   : "habitat",
    "Habitat.Density"           : "habitat_density",
    "Migration"                 : "migration",
    "Trophic.Level"             : "trophic_level",
    "Trophic.Niche"             : "trophic_niche",
    "Primary.Lifestyle"         : "lifestyle",

    # ── Geography
    "Min.Latitude"              : "lat_min",
    "Max.Latitude"              : "lat_max",
    "Centroid.Latitude"         : "lat_centroid",
    "Centroid.Longitude"        : "lon_centroid",
    "Range.Size"                : "range_size",
}, inplace=True)

# Verify final column names
print(output.columns.tolist())

['avibase_id', 'beak_culmen_avg', 'beak_culmen_avg_m', 'beak_culmen_avg_f', 'beak_nares_avg', 'beak_nares_avg_m', 'beak_nares_avg_f', 'beak_width_avg', 'beak_width_avg_m', 'beak_width_avg_f', 'beak_depth_avg', 'beak_depth_avg_m', 'beak_depth_avg_f', 'tarsus_avg', 'tarsus_avg_m', 'tarsus_avg_f', 'wing_len_avg', 'wing_len_avg_m', 'wing_len_avg_f', 'kipps_avg', 'kipps_avg_m', 'kipps_avg_f', 'secondary_avg', 'secondary_avg_m', 'secondary_avg_f', 'hwi_avg', 'hwi_avg_m', 'hwi_avg_f', 'tail_avg', 'tail_avg_m', 'tail_avg_f', 'total_individuals', 'female_count', 'male_count', 'mass_avg', 'mass_source', 'inference', 'habitat', 'habitat_density', 'migration', 'trophic_level', 'trophic_niche', 'lifestyle', 'lat_min', 'lat_max', 'lat_centroid', 'lon_centroid', 'range_size']


In [38]:
output.head()

,avibase_id,beak_culmen_avg,beak_culmen_avg_m,beak_culmen_avg_f,beak_nares_avg,beak_nares_avg_m,beak_nares_avg_f,beak_width_avg,beak_width_avg_m,beak_width_avg_f,beak_depth_avg,beak_depth_avg_m,beak_depth_avg_f,tarsus_avg,tarsus_avg_m,tarsus_avg_f,wing_len_avg,wing_len_avg_m,wing_len_avg_f,kipps_avg,kipps_avg_m,kipps_avg_f,secondary_avg,secondary_avg_m,secondary_avg_f,hwi_avg,hwi_avg_m,hwi_avg_f,tail_avg,tail_avg_m,tail_avg_f,total_individuals,female_count,male_count,mass_avg,mass_source,inference,habitat,habitat_density,migration,trophic_level,trophic_niche,lifestyle,lat_min,lat_max,lat_centroid,lon_centroid,range_size
0,AVIBASE-B3F5E5E2,13.000000,12.133333,14.30,9.500,9.00,10.000000,1.1800,1.233333,1.100000,1.680,1.633333,1.750000,6.340000,5.533333,7.550000,47.000000,48.333333,45.000000,30.500,30.5,30.500000,15.500,16.50,14.50,66.350,64.90,67.800000,29.400,30.000000,28.500000,5.0,2.0,3.0,2.7,Dunning,NO,Forest,2.0,1.0,Herbivore,Nectarivore,Aerial,12.78,19.77,15.29,-89.79,144610.40
1,AVIBASE-684016CB,8.980000,9.066667,8.85,4.900,4.90,4.900000,2.7400,2.666667,2.850000,2.340,2.366667,2.300000,15.280000,15.433333,15.050000,43.400000,44.166667,42.250000,5.700,5.8,5.600000,37.300,37.95,36.65,13.275,13.25,13.300000,38.700,38.833333,38.500000,5.0,2.0,3.0,4.8,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,14.97,31.56,24.78,105.57,1704650.60
2,AVIBASE-0FD03AFF,8.950000,8.800000,9.00,4.675,4.50,4.733333,2.6000,2.700000,2.566667,2.325,2.500000,2.266667,16.525000,16.800000,16.433333,47.125000,48.000000,46.833333,7.600,8.4,7.333333,39.525,39.60,39.50,16.125,17.50,15.666667,47.500,48.000000,47.333333,4.0,3.0,1.0,4.7,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,20.19,31.12,26.70,95.58,909217.05
3,AVIBASE-A8F5D21F,12.320000,12.266667,12.40,7.375,7.35,7.400000,3.1200,3.100000,3.150000,2.840,2.966667,2.650000,16.580000,16.766667,16.300000,51.200000,52.333333,49.500000,6.475,6.7,6.250000,44.525,45.80,43.25,12.700,12.75,12.650000,42.400,44.333333,39.500000,5.0,2.0,3.0,6.5,Dunning,NO,Shrubland,1.0,1.0,Carnivore,Invertivore,Insessorial,-8.11,28.81,9.89,105.20,2493437.47
4,AVIBASE-50A42775,37.116667,38.883333,35.35,16.625,17.20,16.050000,9.2625,9.275000,9.250000,10.250,11.300000,9.200000,65.816667,68.500000,63.133333,337.181818,343.400000,332.000000,52.500,52.5,52.500000,283.100,291.50,277.50,15.660,15.30,15.900000,290.375,300.333333,280.416667,12.0,6.0,6.0,1405.1,Dunning,NO,Forest,1.0,1.0,Herbivore,Frugivore,Insessorial,-13.43,11.15,-2.79,-75.23,139314.36


---
## 6 — Build Full Species Identification Table

We want a table with one row per `Avibase.ID` containing the species name and taxonomy (family + order) from **all three** source taxonomies: BirdLife, eBird, and BirdTree.

- **BirdLife** — matched via `Avibase.ID` directly
- **eBird** — also has `Avibase.ID` (stored as `Avibase.ID2`)
- **BirdTree** — has **no `Avibase.ID`**; matched via species name (`Species3_BirdTree`)

We already have the species names from the raw data deduplication in Section 2. Now we add family and order from each taxonomy.

### 6a — Add BirdLife Family & Order

In [39]:
# BirdLife already loaded in Section 4
# Merge Family1 and Order1 into the identification table
for col in ["Family1", "Order1"]:
    df_Identification = merge_by_avibase(df_Identification, df_birdLife, col)

df_Identification.head()

,Avibase.ID,Species1_BirdLife,Species2_eBird,Species3_BirdTree,Family1,Order1
0,AVIBASE-000E026B,Otus everetti,Otus everetti,Otus megalotis,Strigidae,Strigiformes
1,AVIBASE-00124D98,Zenaida macroura,Zenaida macroura,Zenaida macroura,Columbidae,Columbiformes
2,AVIBASE-00146574,None,Treron calvus,Treron calvus,NaN,NaN
3,AVIBASE-0018A99B,Bubo cinerascens,Bubo cinerascens,Bubo cinerascens,Strigidae,Strigiformes
4,AVIBASE-002105C7,Treron sieboldii,Treron sieboldii,Treron sieboldii,Columbidae,Columbiformes


### 6b — Add eBird Family & Order

In [40]:
df_ebird = pd.read_excel(BASE / "core/AVONET2_eBird.xlsx", sheet_name=1, header=0)

# Standardise the join key name
df_ebird.rename(columns={"Avibase.ID2": "Avibase.ID"}, inplace=True)

print(f"eBird shape: {df_ebird.shape}")
df_ebird.head()

eBird shape: (10661, 31)


,Species2,Family2,Order2,Avibase.ID,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,Beak.Width,Beak.Depth,Tarsus.Length,Wing.Length,Kipps.Distance,Secondary1,Hand-Wing.Index,Tail.Length,Mass,Mass.Source,Mass.Refs.Other,Inference,Traits.inferred,Reference.species,Habitat,Habitat.Density,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle
0,Accipiter albogularis,Accipitridae,Accipitriformes,AVIBASE-BBB59880,5,2,0,3,4,27.7,17.8,10.6,14.7,62.0,235.2,81.8,159.5,33.9,169.0,248.8,Dunning,NaN,NO,NaN,NaN,Forest,1,2.0,Carnivore,Vertivore,Insessorial
1,Accipiter badius,Accipitridae,Accipitriformes,AVIBASE-1A0ECB6E,10,4,6,0,8,20.6,12.1,8.8,11.6,43.0,186.7,62.5,127.4,32.9,140.6,131.2,Dunning,NaN,NO,NaN,NaN,Shrubland,2,3.0,Carnivore,Vertivore,Insessorial
2,Accipiter bicolor,Accipitridae,Accipitriformes,AVIBASE-ADBE44E1,11,4,5,2,8,25.0,13.7,8.6,12.7,58.1,229.6,56.6,174.8,24.6,186.3,287.5,Dunning,NaN,NO,NaN,NaN,Woodland,2,2.0,Carnivore,Vertivore,Generalist
3,Accipiter brachyurus,Accipitridae,Accipitriformes,AVIBASE-68BF920B,4,4,0,0,3,22.5,14.0,8.9,11.9,61.2,202.2,64.1,138.1,31.7,140.8,142.0,Dunning,NaN,NO,NaN,NaN,Forest,1,2.0,Carnivore,Vertivore,Insessorial
4,Accipiter brevipes,Accipitridae,Accipitriformes,AVIBASE-8492E4B7,8,4,4,0,4,21.1,12.1,8.7,11.1,46.4,217.6,87.8,129.9,40.2,153.5,186.5,Dunning,NaN,NO,NaN,NaN,Forest,1,3.0,Carnivore,Vertivore,Generalist


In [41]:
for col in ["Family2", "Order2"]:
    df_Identification = merge_by_avibase(df_Identification, df_ebird, col)

df_Identification.head()

,Avibase.ID,Species1_BirdLife,Species2_eBird,Species3_BirdTree,Family1,Order1,Family2,Order2
0,AVIBASE-000E026B,Otus everetti,Otus everetti,Otus megalotis,Strigidae,Strigiformes,Strigidae,Strigiformes
1,AVIBASE-00124D98,Zenaida macroura,Zenaida macroura,Zenaida macroura,Columbidae,Columbiformes,Columbidae,Columbiformes
2,AVIBASE-00146574,None,Treron calvus,Treron calvus,NaN,NaN,Columbidae,Columbiformes
3,AVIBASE-0018A99B,Bubo cinerascens,Bubo cinerascens,Bubo cinerascens,Strigidae,Strigiformes,Strigidae,Strigiformes
4,AVIBASE-002105C7,Treron sieboldii,Treron sieboldii,Treron sieboldii,Columbidae,Columbiformes,Columbidae,Columbiformes


### 6c — Add BirdTree Family & Order

BirdTree uses its own species names (`Species3`) and has no `Avibase.ID`, so we match on the species name column instead.

The crosswalk file (`BirdLife_BirdTree_Taxonomy_Relation.csv`) maps BirdLife names to BirdTree names — but we already have `Species3_BirdTree` in our raw data, so we can join directly on that.

In [42]:
df_birdtree = pd.read_excel(BASE / "core/AVONET3_BirdTree.xlsx", sheet_name=1, header=0)

print(f"BirdTree shape: {df_birdtree.shape}")
df_birdtree.head()

BirdTree shape: (9993, 36)


,Species3,Family3,Order3,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,Beak.Width,Beak.Depth,Tarsus.Length,Wing.Length,Kipps.Distance,Secondary1,Hand-Wing.Index,Tail.Length,Mass,Mass.Source,Mass.Refs.Other,Inference,Traits.inferred,Reference.species,Habitat,Habitat.Density,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Min.Latitude,Max.Latitude,Centroid.Latitude,Centroid.Longitude,Range.Size,Species.Status
0,Accipiter albogularis,Accipitridae,Accipitriformes,5,2,0,3,4,27.7,17.8,10.6,14.7,62.0,235.2,81.8,159.5,33.9,169.0,248.75,Dunning,NaN,NO,NaN,NaN,Forest,1.0,2.0,Carnivore,Vertivore,Insessorial,-11.73,-4.02,-8.15,158.493765,37461.21,Extant
1,Accipiter badius,Accipitridae,Accipitriformes,10,4,6,0,8,20.6,12.1,8.8,11.6,43.0,186.7,62.5,127.4,32.9,140.6,131.15,Dunning,NaN,NO,NaN,NaN,Shrubland,2.0,3.0,Carnivore,Vertivore,Insessorial,-29.47,46.39,8.23,44.982464,22374973.00,Extant
2,Accipiter bicolor,Accipitridae,Accipitriformes,6,2,2,2,4,26.5,14.8,9.2,13.5,57.5,231.8,46.4,189.6,19.8,188.4,287.54,Dunning,NaN,NO,NaN,NaN,Woodland,2.0,2.0,Carnivore,Vertivore,Generalist,NaN,NaN,NaN,NaN,NaN,Extant
3,Accipiter brachyurus,Accipitridae,Accipitriformes,4,4,0,0,3,22.5,14.0,8.9,11.9,61.2,202.2,64.1,138.1,31.7,140.8,142.00,Dunning,NaN,NO,NaN,NaN,Forest,1.0,2.0,Carnivore,Vertivore,Insessorial,-6.31,-4.08,-5.45,150.681314,35580.71,Extant
4,Accipiter brevipes,Accipitridae,Accipitriformes,8,4,4,0,4,21.1,12.1,8.7,11.1,46.4,217.6,87.8,129.9,40.2,153.5,186.48,Dunning,NaN,NO,NaN,NaN,Forest,1.0,3.0,Carnivore,Vertivore,Generalist,31.19,55.86,45.24,45.327340,2936751.80,Extant


In [43]:
def merge_by_species3(our_dataset, birdtree_dataset, feature):
    """
    Left-join a single feature from BirdTree into our dataset.

    BirdTree has no Avibase.ID, so we match on species name:
      our_dataset["Species3_BirdTree"]  ←→  birdtree_dataset["Species3"]

    The redundant "Species3" key column is dropped after joining.

    Parameters
    ----------
    our_dataset      : main dataframe (must have Species3_BirdTree column)
    birdtree_dataset : BirdTree dataframe (must have Species3 column)
    feature          : column name to bring in

    Returns
    -------
    our_dataset with the new column appended
    """
    result = our_dataset.merge(
        birdtree_dataset[["Species3", feature]],
        left_on="Species3_BirdTree",
        right_on="Species3",
        how="left"
    )
    result.drop(columns=["Species3"], inplace=True)
    return result

In [44]:
for col in ["Family3", "Order3"]:
    df_Identification = merge_by_species3(df_Identification, df_birdtree, col)

df_Identification.head()

,Avibase.ID,Species1_BirdLife,Species2_eBird,Species3_BirdTree,Family1,Order1,Family2,Order2,Family3,Order3
0,AVIBASE-000E026B,Otus everetti,Otus everetti,Otus megalotis,Strigidae,Strigiformes,Strigidae,Strigiformes,Strigidae,Strigiformes
1,AVIBASE-00124D98,Zenaida macroura,Zenaida macroura,Zenaida macroura,Columbidae,Columbiformes,Columbidae,Columbiformes,Columbidae,Columbiformes
2,AVIBASE-00146574,None,Treron calvus,Treron calvus,NaN,NaN,Columbidae,Columbiformes,Columbidae,Columbiformes
3,AVIBASE-0018A99B,Bubo cinerascens,Bubo cinerascens,Bubo cinerascens,Strigidae,Strigiformes,Strigidae,Strigiformes,Strigidae,Strigiformes
4,AVIBASE-002105C7,Treron sieboldii,Treron sieboldii,Treron sieboldii,Columbidae,Columbiformes,Columbidae,Columbiformes,Columbidae,Columbiformes


### 6d — Rename Identification Table Columns

In [45]:
df_Identification.rename(columns={
    "Avibase.ID"        : "avibase_id",

    # ── BirdLife (1)
    "Species1_BirdLife" : "species_birdlife",
    "Family1"           : "family_birdlife",
    "Order1"            : "order_birdlife",

    # ── eBird (2)
    "Species2_eBird"    : "species_ebird",
    "Family2"           : "family_ebird",
    "Order2"            : "order_ebird",

    # ── BirdTree (3)
    "Species3_BirdTree" : "species_birdtree",
    "Family3"           : "family_birdtree",
    "Order3"            : "order_birdtree",
}, inplace=True)

print(f"Shape: {df_Identification.shape}")
df_Identification.head()

Shape: (11237, 10)


,avibase_id,species_birdlife,species_ebird,species_birdtree,family_birdlife,order_birdlife,family_ebird,order_ebird,family_birdtree,order_birdtree
0,AVIBASE-000E026B,Otus everetti,Otus everetti,Otus megalotis,Strigidae,Strigiformes,Strigidae,Strigiformes,Strigidae,Strigiformes
1,AVIBASE-00124D98,Zenaida macroura,Zenaida macroura,Zenaida macroura,Columbidae,Columbiformes,Columbidae,Columbiformes,Columbidae,Columbiformes
2,AVIBASE-00146574,None,Treron calvus,Treron calvus,NaN,NaN,Columbidae,Columbiformes,Columbidae,Columbiformes
3,AVIBASE-0018A99B,Bubo cinerascens,Bubo cinerascens,Bubo cinerascens,Strigidae,Strigiformes,Strigidae,Strigiformes,Strigidae,Strigiformes
4,AVIBASE-002105C7,Treron sieboldii,Treron sieboldii,Treron sieboldii,Columbidae,Columbiformes,Columbidae,Columbiformes,Columbidae,Columbiformes


---
## 7 — Final Summary

We now have two clean output tables:

| Table | Rows | Description |
|---|---|---|
| `df_Identification` | 11,237 | One row per species with names and taxonomy from all 3 sources |
| `output` | 11,237 | One row per species with morphological averages (overall, M, F) + ecology data |

In [46]:
print("=== df_Identification ===")
print(f"Rows    : {len(df_Identification):,}")
print(f"Columns : {df_Identification.columns.tolist()}")
print()
print("=== output (morphology + ecology) ===")
print(f"Rows    : {len(output):,}")
print(f"Columns : {len(output.columns)}")

=== df_Identification ===
Rows    : 11,237
Columns : ['avibase_id', 'species_birdlife', 'species_ebird', 'species_birdtree', 'family_birdlife', 'order_birdlife', 'family_ebird', 'order_ebird', 'family_birdtree', 'order_birdtree']

=== output (morphology + ecology) ===
Rows    : 11,237
Columns : 48


In [47]:
output

,avibase_id,beak_culmen_avg,beak_culmen_avg_m,beak_culmen_avg_f,beak_nares_avg,beak_nares_avg_m,beak_nares_avg_f,beak_width_avg,beak_width_avg_m,beak_width_avg_f,beak_depth_avg,beak_depth_avg_m,beak_depth_avg_f,tarsus_avg,tarsus_avg_m,tarsus_avg_f,wing_len_avg,wing_len_avg_m,wing_len_avg_f,kipps_avg,kipps_avg_m,kipps_avg_f,secondary_avg,secondary_avg_m,secondary_avg_f,hwi_avg,hwi_avg_m,hwi_avg_f,tail_avg,tail_avg_m,tail_avg_f,total_individuals,female_count,male_count,mass_avg,mass_source,inference,habitat,habitat_density,migration,trophic_level,trophic_niche,lifestyle,lat_min,lat_max,lat_centroid,lon_centroid,range_size
0,AVIBASE-B3F5E5E2,13.000000,12.133333,14.30,9.500000,9.000000,10.000000,1.180000,1.233333,1.100000,1.680000,1.633333,1.750000,6.340000,5.533333,7.550000,47.000000,48.333333,45.000000,30.500000,30.500000,30.500000,15.500000,16.500000,14.50,66.350000,64.900000,67.800000,29.400000,30.000000,28.500000,5.0,2.0,3.0,2.7,Dunning,NO,Forest,2.0,1.0,Herbivore,Nectarivore,Aerial,12.78,19.77,15.29,-89.79,144610.40
1,AVIBASE-684016CB,8.980000,9.066667,8.85,4.900000,4.900000,4.900000,2.740000,2.666667,2.850000,2.340000,2.366667,2.300000,15.280000,15.433333,15.050000,43.400000,44.166667,42.250000,5.700000,5.800000,5.600000,37.300000,37.950000,36.65,13.275000,13.250000,13.300000,38.700000,38.833333,38.500000,5.0,2.0,3.0,4.8,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,14.97,31.56,24.78,105.57,1704650.60
2,AVIBASE-0FD03AFF,8.950000,8.800000,9.00,4.675000,4.500000,4.733333,2.600000,2.700000,2.566667,2.325000,2.500000,2.266667,16.525000,16.800000,16.433333,47.125000,48.000000,46.833333,7.600000,8.400000,7.333333,39.525000,39.600000,39.50,16.125000,17.500000,15.666667,47.500000,48.000000,47.333333,4.0,3.0,1.0,4.7,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,20.19,31.12,26.70,95.58,909217.05
3,AVIBASE-A8F5D21F,12.320000,12.266667,12.40,7.375000,7.350000,7.400000,3.120000,3.100000,3.150000,2.840000,2.966667,2.650000,16.580000,16.766667,16.300000,51.200000,52.333333,49.500000,6.475000,6.700000,6.250000,44.525000,45.800000,43.25,12.700000,12.750000,12.650000,42.400000,44.333333,39.500000,5.0,2.0,3.0,6.5,Dunning,NO,Shrubland,1.0,1.0,Carnivore,Invertivore,Insessorial,-8.11,28.81,9.89,105.20,2493437.47
4,AVIBASE-50A42775,37.116667,38.883333,35.35,16.625000,17.200000,16.050000,9.262500,9.275000,9.250000,10.250000,11.300000,9.200000,65.816667,68.500000,63.133333,337.181818,343.400000,332.000000,52.500000,52.500000,52.500000,283.100000,291.500000,277.50,15.660000,15.300000,15.900000,290.375000,300.333333,280.416667,12.0,6.0,6.0,1405.1,Dunning,NO,Forest,1.0,1.0,Herbivore,Frugivore,Insessorial,-13.43,11.15,-2.79,-75.23,139314.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11232,AVIBASE-4D7B5D9D,17.700000,18.033333,17.20,9.700000,9.966667,9.300000,3.820000,3.966667,3.600000,5.000000,4.833333,5.250000,22.620000,23.300000,21.600000,71.480000,73.300000,68.750000,12.560000,12.633333,12.450000,58.920000,60.666667,56.30,17.580000,17.233333,18.100000,60.340000,61.233333,59.000000,5.0,2.0,3.0,21.6,Inferred,YES,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,8.76,9.50,9.08,117.96,256.97
11233,AVIBASE-E87556DB,21.000000,21.000000,NaN,11.400000,11.400000,NaN,4.000000,4.000000,NaN,5.200000,5.200000,NaN,24.900000,24.900000,NaN,67.000000,67.000000,NaN,8.400000,8.400000,NaN,58.600000,58.600000,NaN,12.500000,12.500000,NaN,59.000000,59.000000,NaN,1.0,0.0,1.0,29.5,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,10.95,11.43,11.21,122.22,285.09
11234,AVIBASE-2344C21D,15.222222,16.385714,11.15,8.800000,9.214286,7.350000,3.388889,3.557143,2.800000,3.777778,3.971429,3.100000,22.288889,23.500000,18.050000,68.200000,72.342857,53.700000,10.866667,11.000000,10.400000,57.333333,61.342857,43.30,16.133333,15.228571,19.300000,49.555556,54.357143,32.750000,9.0,2.0,7.0,21.6,Dunning,NO

In [48]:
avonet_uncleaned_final_dataset = output

In [57]:
avonet_uncleaned_final_dataset.head()

,avibase_id,beak_culmen_avg,beak_culmen_avg_m,beak_culmen_avg_f,beak_nares_avg,beak_nares_avg_m,beak_nares_avg_f,beak_width_avg,beak_width_avg_m,beak_width_avg_f,beak_depth_avg,beak_depth_avg_m,beak_depth_avg_f,tarsus_avg,tarsus_avg_m,tarsus_avg_f,wing_len_avg,wing_len_avg_m,wing_len_avg_f,kipps_avg,kipps_avg_m,kipps_avg_f,secondary_avg,secondary_avg_m,secondary_avg_f,hwi_avg,hwi_avg_m,hwi_avg_f,tail_avg,tail_avg_m,tail_avg_f,total_individuals,female_count,male_count,mass_avg,mass_source,inference,habitat,habitat_density,migration,trophic_level,trophic_niche,lifestyle,lat_min,lat_max,lat_centroid,lon_centroid,range_size
0,AVIBASE-B3F5E5E2,13.000000,12.133333,14.30,9.500,9.00,10.000000,1.1800,1.233333,1.100000,1.680,1.633333,1.750000,6.340000,5.533333,7.550000,47.000000,48.333333,45.000000,30.500,30.5,30.500000,15.500,16.50,14.50,66.350,64.90,67.800000,29.400,30.000000,28.500000,5.0,2.0,3.0,2.7,Dunning,NO,Forest,2.0,1.0,Herbivore,Nectarivore,Aerial,12.78,19.77,15.29,-89.79,144610.40
1,AVIBASE-684016CB,8.980000,9.066667,8.85,4.900,4.90,4.900000,2.7400,2.666667,2.850000,2.340,2.366667,2.300000,15.280000,15.433333,15.050000,43.400000,44.166667,42.250000,5.700,5.8,5.600000,37.300,37.95,36.65,13.275,13.25,13.300000,38.700,38.833333,38.500000,5.0,2.0,3.0,4.8,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,14.97,31.56,24.78,105.57,1704650.60
2,AVIBASE-0FD03AFF,8.950000,8.800000,9.00,4.675,4.50,4.733333,2.6000,2.700000,2.566667,2.325,2.500000,2.266667,16.525000,16.800000,16.433333,47.125000,48.000000,46.833333,7.600,8.4,7.333333,39.525,39.60,39.50,16.125,17.50,15.666667,47.500,48.000000,47.333333,4.0,3.0,1.0,4.7,Dunning,NO,Forest,1.0,1.0,Carnivore,Invertivore,Insessorial,20.19,31.12,26.70,95.58,909217.05
3,AVIBASE-A8F5D21F,12.320000,12.266667,12.40,7.375,7.35,7.400000,3.1200,3.100000,3.150000,2.840,2.966667,2.650000,16.580000,16.766667,16.300000,51.200000,52.333333,49.500000,6.475,6.7,6.250000,44.525,45.80,43.25,12.700,12.75,12.650000,42.400,44.333333,39.500000,5.0,2.0,3.0,6.5,Dunning,NO,Shrubland,1.0,1.0,Carnivore,Invertivore,Insessorial,-8.11,28.81,9.89,105.20,2493437.47
4,AVIBASE-50A42775,37.116667,38.883333,35.35,16.625,17.20,16.050000,9.2625,9.275000,9.250000,10.250,11.300000,9.200000,65.816667,68.500000,63.133333,337.181818,343.400000,332.000000,52.500,52.5,52.500000,283.100,291.50,277.50,15.660,15.30,15.900000,290.375,300.333333,280.416667,12.0,6.0,6.0,1405.1,Dunning,NO,Forest,1.0,1.0,Herbivore,Frugivore,Insessorial,-13.43,11.15,-2.79,-75.23,139314.36


In [ ]:
avonet_uncleaned_final_dataset.to_csv(
    'avonet_uncleaned_final_dataset.csv',
    index=False,               
    na_rep='MISSING',   
    float_format='%.5f' 
)

df_Identification.to_csv(
    'avonet_crosswalk.csv',
    index=False,               
    na_rep='MISSING',   
)

In [ ]:
df_new1 = pd.read_csv("avonet_uncleaned_final_dataset.csv")
df_new2 = pd.read_csv("avonet_crosswalk.csv")

In [ ]:
df_new1.head()

,avibase_id,beak_culmen_avg,beak_culmen_avg_m,beak_culmen_avg_f,beak_nares_avg,beak_nares_avg_m,beak_nares_avg_f,beak_width_avg,beak_width_avg_m,beak_width_avg_f,beak_depth_avg,beak_depth_avg_m,beak_depth_avg_f,tarsus_avg,tarsus_avg_m,tarsus_avg_f,wing_len_avg,wing_len_avg_m,wing_len_avg_f,kipps_avg,kipps_avg_m,kipps_avg_f,secondary_avg,secondary_avg_m,secondary_avg_f,hwi_avg,hwi_avg_m,hwi_avg_f,tail_avg,tail_avg_m,tail_avg_f,total_individuals,female_count,male_count,mass_avg,mass_source,inference,habitat,habitat_density,migration,trophic_level,trophic_niche,lifestyle,lat_min,lat_max,lat_centroid,lon_centroid,range_size
0,AVIBASE-B3F5E5E2,13.00000,12.13333,14.30000,9.50000,9.00000,10.00000,1.18000,1.23333,1.10000,1.68000,1.63333,1.75000,6.34000,5.53333,7.55000,47.00000,48.33333,45.00000,30.50000,30.50000,30.50000,15.50000,16.50000,14.50000,66.35000,64.90000,67.80000,29.40000,30.00000,28.50000,5.00000,2.00000,3.00000,2.70000,Dunning,NO,Forest,2.00000,1.00000,Herbivore,Nectarivore,Aerial,12.78000,19.77000,15.29000,-89.79000,144610.40000
1,AVIBASE-684016CB,8.98000,9.06667,8.85000,4.90000,4.90000,4.90000,2.74000,2.66667,2.85000,2.34000,2.36667,2.30000,15.28000,15.43333,15.05000,43.40000,44.16667,42.25000,5.70000,5.80000,5.60000,37.30000,37.95000,36.65000,13.27500,13.25000,13.30000,38.70000,38.83333,38.50000,5.00000,2.00000,3.00000,4.80000,Dunning,NO,Forest,1.00000,1.00000,Carnivore,Invertivore,Insessorial,14.97000,31.56000,24.78000,105.57000,1704650.60000
2,AVIBASE-0FD03AFF,8.95000,8.80000,9.00000,4.67500,4.50000,4.73333,2.60000,2.70000,2.56667,2.32500,2.50000,2.26667,16.52500,16.80000,16.43333,47.12500,48.00000,46.83333,7.60000,8.40000,7.33333,39.52500,39.60000,39.50000,16.12500,17.50000,15.66667,47.50000,48.00000,47.33333,4.00000,3.00000,1.00000,4.70000,Dunning,NO,Forest,1.00000,1.00000,Carnivore,Invertivore,Insessorial,20.19000,31.12000,26.70000,95.58000,909217.05000
3,AVIBASE-A8F5D21F,12.32000,12.26667,12.40000,7.37500,7.35000,7.40000,3.12000,3.10000,3.15000,2.84000,2.96667,2.65000,16.58000,16.76667,16.30000,51.20000,52.33333,49.50000,6.47500,6.70000,6.25000,44.52500,45.80000,43.25000,12.70000,12.75000,12.65000,42.40000,44.33333,39.50000,5.00000,2.00000,3.00000,6.50000,Dunning,NO,Shrubland,1.00000,1.00000,Carnivore,Invertivore,Insessorial,-8.11000,28.81000,9.89000,105.20000,2493437.47000
4,AVIBASE-50A42775,37.11667,38.88333,35.35000,16.62500,17.20000,16.05000,9.26250,9.27500,9.25000,10.25000,11.30000,9.20000,65.81667,68.50000,63.13333,337.18182,343.40000,332.00000,52.50000,52.50000,52.50000,283.10000,291.50000,277.50000,15.66000,15.30000,15.90000,290.37500,300.33333,280.41667,12.00000,6.00000,6.00000,1405.10000,Dunning,NO,Forest,1.00000,1.00000,Herbivore,Frugivore,Insessorial,-13.43000,11.15000,-2.79000,-75.23000,139314.36000


In [ ]:
df_new2.head()